# DanceMatch3 — kostenloser Trainingslauf in Google Colab

Dieses Notebook lädt die vorhandene Baseline und startet das Training in der kostenlosen Colab-Runtime. Kein GitHub-Actions-Lauf und kein GitHub-Billing nötig.

**Kein manueller Musik-Upload:** Vor dem Training versucht das Notebook, die im Repository dokumentierten, offen lizenzierten Zusatztracks automatisch herunterzuladen. Aktuell sind das nur ein Bachata- und ein Salsa-Titel. Das ist ein Start, aber kein ausreichender Datensatz für robuste Erkennung; Discofox ist weiterhin nicht durch eine passende, verifizierte Audioquelle abgedeckt.

Das Modell ist ein Forschungs-Baseline-Test, noch kein produktionsreifes Modell für alle neun Tanzstile. Lizenzbedingungen der Musikquellen vor einer Produktverwendung prüfen.

In [ ]:
%cd /content
import os, pathlib, shutil
os.chdir('/content')
os.environ['PWD'] = '/content'
REPO_PATH = pathlib.Path('/content/DanceMatch3')
CACHE_DIR = pathlib.Path('/content/dancematch_hf_cache')

# Alte DanceMatch-/MuCUE-Caches aus einem abgebrochenen, nicht gestreamten Lauf gezielt entfernen.
if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
legacy_dataset_cache = pathlib.Path.home() / '.cache' / 'huggingface' / 'datasets'
if legacy_dataset_cache.exists():
    for p in legacy_dataset_cache.glob('*MuCUE*'):
        shutil.rmtree(p, ignore_errors=True)
legacy_hub_cache = pathlib.Path.home() / '.cache' / 'huggingface' / 'hub' / 'datasets--Yi3852--MuCUE'
if legacy_hub_cache.exists():
    shutil.rmtree(legacy_hub_cache, ignore_errors=True)

# Alte Installations-Caches vor dem Start entfernen.
!apt-get clean
!pip cache purge >/dev/null 2>&1 || true
total, used, free = shutil.disk_usage('/content')
print(f'Freier Colab-Speicher vor Installation: {free / (1024**3):.2f} GiB von {total / (1024**3):.2f} GiB')
if free < 1.5 * (1024**3):
    raise RuntimeError('Weniger als 1.5 GiB frei. Die Runtime wurde vorsorglich gestoppt, damit sie nicht voll läuft. Bitte Colab-Runtime neu starten und erneut ausführen.')

# Systembibliotheken und kompatible Python-Abhängigkeiten installieren.
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1
!pip -q install 'datasets>=3.0,<4' 'librosa>=0.10.2,<1' 'numpy>=1.26,<2.3' 'scikit-learn>=1.4,<2' 'joblib>=1.3,<2' 'fsspec==2025.3.0' 'gcsfs==2025.3.0'
import datasets, fsspec
print('datasets:', datasets.__version__, '| fsspec:', fsspec.__version__)
try:
    import gcsfs
    print('gcsfs:', gcsfs.__version__)
except Exception as e:
    print('gcsfs optional nicht verfügbar:', e)
!apt-get clean
!rm -rf /var/lib/apt/lists/* /var/cache/apt/archives/*
!pip cache purge >/dev/null 2>&1 || true
total, used, free = shutil.disk_usage('/content')
print(f'Colab-Disk nach Setup: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')
print('Setup abgeschlossen')

In [ ]:
# Git-Befehle mit sauberem Python- und Shell-Arbeitsverzeichnis ausführen.
%cd /content
import os, pathlib, shutil, subprocess
BASE = '/content'
REPO = os.path.join(BASE, 'DanceMatch3')
BRANCH = 'feature/dance-style-training'
URL = 'https://github.com/zwirbel/DanceMatch3.git'
os.chdir(BASE)
os.environ['PWD'] = BASE

if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print(f'{REPO} existiert, ist aber kein Git-Repository. Entferne es ...')
    shutil.rmtree(REPO)

if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], cwd=BASE, check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], cwd=BASE, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, URL, REPO], cwd=BASE, check=True)

script = pathlib.Path(REPO, 'training', 'train_baseline.py')
if not script.is_file():
    raise FileNotFoundError(f'Trainingsskript nicht gefunden: {script}')
print('Code erfolgreich geladen:', REPO)
print('Branch:', subprocess.run(['git', '-C', REPO, 'branch', '--show-current'], cwd=BASE, check=True, capture_output=True, text=True).stdout.strip())


## Trainingsumfang auswählen

Der Basisdatensatz wird **gestreamt**: Audiodaten werden einzeln abgerufen und verarbeitet, statt den gesamten Datensatz als lokale Dataset-Kopie zu speichern. Vor dem Training werden zusätzliche, im Repository dokumentierte Audioquellen automatisch heruntergeladen; du musst kein ZIP erstellen oder hochladen.

MAX_ITEMS = 0 verarbeitet alle verfügbaren Beispiele (bis zu 698 Zeilen laut aktuellem Datensatzstand). Eine positive Zahl, zum Beispiel 100, begrenzt den Lauf auf einen kurzen Funktionstest. Das Skript bricht vorsorglich ab, wenn weniger als 0.75 GiB freier Speicher vorhanden ist.

In [ ]:
MAX_ITEMS = 0  # 0 = vollständiger Streaming-Lauf; positive Zahl = kurzer Funktionstest
DATASET = 'Yi3852/MuCUE'
SPLIT = 'ballroom_genres'
CACHE_DIR = '/content/dancematch_hf_cache'
OUT_DIR = f'{REPO}/training/artifacts'
SUPPLEMENT_DIR = f'{REPO}/training/supplemental_audio'
os.makedirs(OUT_DIR, exist_ok=True)
print('MAX_ITEMS =', MAX_ITEMS, '| DATASET =', DATASET, '| SPLIT =', SPLIT)
print('Zusatzmusik wird automatisch geladen aus:', SUPPLEMENT_DIR)

In [ ]:
# Zusatzmusik automatisch aus den im Repository dokumentierten Quellen beziehen.
# Kein Upload-Dialog und keine ZIP-Datei erforderlich.
from pathlib import Path
import shutil, subprocess, sys, os

SUPPLEMENT_DIR = Path('/content/DanceMatch3/training/supplemental_audio')
download_script = Path('/content/DanceMatch3/training/prepare_supplemental_audio.py')

if SUPPLEMENT_DIR.exists():
    shutil.rmtree(SUPPLEMENT_DIR, ignore_errors=True)

print('Versuche, die freigegebenen Startquellen automatisch herunterzuladen ...')
result = subprocess.run(
    [sys.executable, str(download_script)],
    cwd=str(download_script.parent),
    check=False,
    text=True
)
if result.returncode != 0:
    print('WARNUNG: Zusatzmusik konnte nicht vollständig geladen werden; das Training versucht mit den verfügbaren Dateien fortzufahren.')

supported = {'.wav', '.mp3', '.m4a', '.ogg', '.oga', '.flac', '.aac', '.webm', '.mp4'}
found = [
    p for p in SUPPLEMENT_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in supported
] if SUPPLEMENT_DIR.exists() else []
counts = {}
for p in found:
    label = p.parent.name
    counts[label] = counts.get(label, 0) + 1
print('Automatisch heruntergeladene Audiodateien:', len(found))
print('Dateien je Klasse:', counts or 'keine')
print('Hinweis: Mit der aktuellen Quellenliste sind Bachata/Salsa nur sehr schwach abgedeckt; Discofox fehlt weiterhin.')

In [ ]:
# Alte Modell- und Bewertungsdateien entfernen, damit ein fehlgeschlagener Lauf
# nicht versehentlich Ergebnisse eines früheren Laufs als aktuell ausgibt.
from pathlib import Path
ARTIFACTS = Path('/content/DanceMatch3/training/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ('dance_baseline.joblib', 'evaluation.json', 'run_manifest.json'):
    old_file = ARTIFACTS / name
    if old_file.exists():
        old_file.unlink()
        print('Altes Ergebnis entfernt:', old_file)

%cd /content/DanceMatch3/training
!DANCEMATCH_CACHE_DIR="$CACHE_DIR" HF_HUB_DISABLE_TELEMETRY=1 python train_baseline.py --dataset "$DATASET" --split "$SPLIT" --max-items "$MAX_ITEMS" --supplement-dir "$SUPPLEMENT_DIR" --out artifacts

# Raw uploaded audio is temporary; keep only the trained model and evaluation report.
supplement_path = pathlib.Path(SUPPLEMENT_DIR)
if supplement_path.exists():
    extra_bytes = sum(p.stat().st_size for p in supplement_path.rglob('*') if p.is_file())
    shutil.rmtree(supplement_path, ignore_errors=True)
    print(f'Zusatz-Audiodaten nach dem Lauf gelöscht: {extra_bytes / (1024**2):.1f} MiB')
total, used, free = shutil.disk_usage('/content')
print(f'Colab-Speicher nach Trainingslauf: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')

In [ ]:
# Nur die Ergebnisse des soeben gestarteten Laufs anzeigen und als eindeutig benannte ZIP exportieren.
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import zipfile
from google.colab import files

artifacts = Path('/content/DanceMatch3/training/artifacts')
model_path = artifacts / 'dance_baseline.joblib'
evaluation_path = artifacts / 'evaluation.json'

# Nicht fortfahren, wenn das Training keine neuen Ergebnisdateien erzeugt hat.
missing = [str(p) for p in (model_path, evaluation_path) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Aktuelle Trainingsergebnisse fehlen: ' + ', '.join(missing) +
        '. Bitte zuerst die Trainingszelle erfolgreich ausführen.'
    )

report = json.loads(evaluation_path.read_text(encoding='utf-8'))
now = datetime.now(timezone.utc)
manifest = {
    'created_at_utc': now.isoformat(timespec='seconds'),
    'model_file': model_path.name,
    'model_size_bytes': model_path.stat().st_size,
    'model_modified_at_utc': datetime.fromtimestamp(
        model_path.stat().st_mtime, timezone.utc
    ).isoformat(timespec='seconds'),
    'evaluation_file': evaluation_path.name,
    'evaluation_size_bytes': evaluation_path.stat().st_size,
    'evaluation_modified_at_utc': datetime.fromtimestamp(
        evaluation_path.stat().st_mtime, timezone.utc
    ).isoformat(timespec='seconds'),
    'dataset': report.get('dataset'),
    'dataset_split': report.get('dataset_split'),
    'usable_tracks': report.get('usable_tracks'),
    'rows_seen': report.get('rows_seen'),
    'skipped_rows': report.get('skipped_rows'),
    'macro_f1': report.get('macro_f1'),
    'class_counts': report.get('class_counts'),
    'missing_target_classes': report.get('missing_target_classes'),
    'is_nine_class_model': report.get('is_nine_class_model'),
    'supplemental_tracks': report.get('supplemental_tracks'),
}
manifest_path = artifacts / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('AKTUELLER TRAININGSLAUF')
print('Datensatz:', manifest['dataset'], '| Split:', manifest['dataset_split'])
print('Verwendete Musikstücke:', manifest['usable_tracks'])
print('Gesehene Zeilen:', manifest['rows_seen'], '| Übersprungen:', manifest['skipped_rows'])
print('Macro-F1:', manifest['macro_f1'])
print('Klassen:', json.dumps(manifest['class_counts'], ensure_ascii=False))
print('Fehlende Zielklassen:', manifest.get('missing_target_classes'))
print('Vollständiges 9-Klassen-Modell:', manifest.get('is_nine_class_model'))
for p in (model_path, evaluation_path, manifest_path):
    print(p, f'({p.stat().st_size:,} bytes)')

stamp = now.strftime('%Y%m%d-%H%M%S')
zip_path = f'/content/dancematch-training-results-{stamp}-UTC.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(model_path, model_path.name)
    z.write(evaluation_path, evaluation_path.name)
    z.write(manifest_path, manifest_path.name)

files.download(zip_path)
print('ZIP erstellt:', zip_path)

# Sicherheitsnetz: der Trainingsprozess bereinigt seinen Cache bereits im finally-Block.
cache_path = Path(CACHE_DIR)
if cache_path.exists():
    cache_bytes = sum(p.stat().st_size for p in cache_path.rglob('*') if p.is_file())
    shutil.rmtree(cache_path, ignore_errors=True)
    print(f'Übrig gebliebenen DanceMatch-Cache bereinigt: {cache_bytes / (1024**2):.1f} MiB.')
total, used, free = shutil.disk_usage('/content')
print(f'Colab-Disk nach Bereinigung: {free / (1024**3):.2f} GiB frei von {total / (1024**3):.2f} GiB')